# Cells

A *cell* is the top-level object you simulate. Every neuron in `braincell` is a
{class}`braincell.Cell`, whose base is {class}`braincell.HHTypedNeuron`. The same
class covers two modeling regimes.

## Point neurons and morphological cells

```{list-table}
:header-rows: 1
:widths: 24 38 38

* -
  - Point neuron: `Cell.point(...)`
  - Morphological cell: `Cell(morphology)`
* - **Geometry**
  - one isopotential compartment, a cylinder
  - a full {class}`~braincell.Morphology`
* - **Spatial structure**
  - voltage is uniform
  - voltage spreads along dendrites/axons
* - **Voltage `V`**
  - one value per neuron, shape `(pop_size,)`
  - one value per control volume, shape `(pop_size, n_cv)`
* - **How you build it**
  - `paint` and `place` declarations, on the whole cell
  - `paint` and `place` declarations, on regions and locations
* - **Default solver**
  - `ralston2`
  - `staggered`
* - **Best for**
  - channel prototyping, point-neuron networks, fitting
  - dendritic computation, distributed channels, synaptic integration
```

Because both are the same class, a model grows from a point neuron into a
morphological one without changing its declarations, and the biophysics you
learn in one transfers directly to the other.

## Point neurons

{meth}`braincell.Cell.point` makes a cell with one well-mixed compartment. It is
short for `braincell.Cell(braincell.Morphology.from_soma(length=..., radius=...),
cv_policy=braincell.Lumped())`. You then paint ion species and channels onto it,
place a stimulus, record and run:

In [1]:
import brainunit as u
import braincell
from braincell import mech

hh = braincell.Cell.point(100, length=10. * u.um, radius=5. * u.um, solver='exp_euler')
hh.paint(
    mech.Ion('SodiumFixed', E=50. * u.mV),
    mech.Channel('Na_HH1952'),
    mech.Ion('PotassiumFixed', E=-77. * u.mV),
    mech.Channel('K_HH1952'),
    mech.Channel('IL', E=-54.387 * u.mV, g_max=0.03 * (u.mS / u.cm ** 2)),
)
hh.place(mech.CurrentClamp(delay=2. * u.ms, durations=10. * u.ms, amplitudes=0.05 * u.nA))
hh.record('v', braincell.observe.state('V'))

result = hh.run(dt=0.025 * u.ms, duration=20. * u.ms)
result.samples['v'].values.shape, hh.V.value.shape

((800, 100), (100,))

- The first argument is the **population size**: `Cell.point(100)` simulates
  100 independent neurons.
- `paint` without a region covers the whole cell; `place` without a location
  uses its only site, the middle of the compartment.
- The geometry (`length`, `radius`; these are the defaults) turns densities into
  totals: a clamp's 0.05 nA is divided by the membrane area, and the
  capacitance is 1 µF/cm² unless you paint a `mech.CableProperty`.
- `solver` names the integrator (see {doc}`integration`).
- Channels use the ion their current depends on (see {doc}`ions_channels`).
- `run` initializes the cell first: the voltage starts at the resting potential
  (or `V_init=`) with every gate at its steady state there.

To drive the cell from a loop of your own, or from `brainpy` projections in a
network, call `cell.update(I_ext)` once per step with a current or a current
density; {doc}`../getting_started/first_steps` shows the loop.

A reusable model is a function that returns the declared cell, or a `Cell`
subclass:

In [2]:
class HH(braincell.Cell):
    def __init__(self, size, solver='exp_euler'):
        super().__init__(
            braincell.Morphology.from_soma(), pop_size=size, cv_policy=braincell.Lumped(), solver=solver,
        )
        self.paint(
            mech.Ion('SodiumFixed', E=50. * u.mV),
            mech.Channel('Na_HH1952'),
            mech.Ion('PotassiumFixed', E=-77. * u.mV),
            mech.Channel('K_HH1952'),
            mech.Channel('IL', E=-54.387 * u.mV, g_max=0.03 * (u.mS / u.cm ** 2)),
        )


HH(10)

Cell(root='soma', n_branches=1, n_paint_rules=6, n_place_rules=0, initialized=False)

### Coming from `SingleCompartment`

`braincell.SingleCompartment`, the earlier point-neuron class, was removed in
braincell 0.3, together with `braincell.MultiCompartment`, an alias of `Cell`
(use `Cell`). A point neuron written with it translates line by line:

```{list-table}
:header-rows: 1
:widths: 50 50

* - `SingleCompartment`
  - `Cell`
* - `class HH(SingleCompartment)` with `super().__init__(size, length=..., radius=..., ...)`
  - `Cell.point(size, length=..., radius=..., ...)`, or a `Cell` subclass
* - `C=1 * u.uF / u.cm**2`
  - `paint(mech.CableProperty(membrane_capacitance=...))`
* - `V_initializer=...`
  - `V_init=...` (a quantity or an initializer)
* - `V_th`, `spk_fun`, `solver`
  - the same names; the default solver is `ralston2` for both
* - `self.na = ion.SodiumFixed(size, E=...)` and `self.na.add(INa=channel.Na_HH1952(size, g_max=...))`
  - `paint(mech.Ion('SodiumFixed', E=...), mech.Channel('Na_HH1952', g_max=...))`
* - `MixIons(self.k, self.ca)` for a channel on two ions
  - paint both ions and the channel; the cell binds it to both
* - a `Channel` subclass of your own, instantiated directly
  - `mech.Channel(MyChannel, ...)`, without registering it
* - `update(I_ext)` in a loop of your own
  - the same, or `place(mech.CurrentClamp(...))`, `record` and `run`
* - brainpy projections with `post=` the neuron
  - the same
* - totals: `C` and `g_max` multiplied by the area
  - densities: `cm` and `g_max` per area; currents are divided by the area
* - after `init_state()`, HH gates start at 0 (call `reset_state()` for steady state)
  - after `init_state()`, gates start at their steady state
* - `V.shape == (size,)`
  - the same
```

## Morphological cells

A {class}`~braincell.Cell` on a morphology is decorated declaratively in the
same way. The pattern is **paint** (distribute a density mechanism over a
region) and **place** (attach a point mechanism at a location):

In [3]:
from braincell.filter import AllRegion, RootLocation, branch_in

# 1. geometry
morpho = braincell.Morphology.from_swc("../../data/morphology/example_tree.swc")
cell = braincell.Cell(morpho)

# 2. passive cable everywhere
cell.paint(AllRegion(), mech.CableProperty(
    resting_potential=-65. * u.mV,
    membrane_capacitance=1.0 * u.uF / u.cm**2,
    axial_resistivity=100. * u.ohm * u.cm,
))

# 3. channels onto regions
cell.paint(AllRegion(), mech.Channel("IL", g_max=0.3 * u.mS / u.cm**2, E=-70. * u.mV))
cell.paint(
    branch_in("type", "soma"),
    mech.Ion("SodiumFixed", E=50. * u.mV),
    mech.Channel("Na_Ba2002", g_max=120. * u.mS / u.cm**2),
)

# 4. stimulus at the soma
cell.place(RootLocation(0.5),
           mech.CurrentClamp(delay=10 * u.ms, durations=50 * u.ms, amplitudes=0.2 * u.nA))
cell.n_cv, cell.solver_name

(9, 'staggered')

`paint` targets a **region** (a set of cable); `place` targets a **locset** (a
ordered point rows). Those selection expressions come from {mod}`braincell.filter` —
see {doc}`regions_locsets`. The mechanisms themselves are covered in
{doc}`mechanisms`. `record` and `run` work as for the point neuron; the
worked multi-compartment notebooks in {doc}`../tutorials/index` go further.

## The shared base: `HHTypedNeuron`

`Cell` inherits from {class}`braincell.HHTypedNeuron`, a Hodgkin–Huxley-style
neuron that owns the integrator and the ion/channel registry. You rarely
instantiate it directly; it is also the `root_type` of channels that need no
ion, such as the leak `IL`.

## See also

- {doc}`morphology` — building the geometry a `Cell` wraps.
- {doc}`discretization` — how a `Cell`'s geometry becomes control volumes.
- {doc}`../tutorials/index` — hands-on tutorials.